# 03 - Machine Learning: prediccion de ventas

Este notebook construye modelos para predecir unidades vendidas por dia, canal y categoria.

Logica de negocio:

- Si podemos anticipar ventas, podemos planificar stock, promociones y objetivos de margen.
- La prediccion no sustituye al dashboard: lo alimenta con una capa anticipativa.
- Comparamos un baseline simple contra modelos de machine learning para comprobar si realmente aportan valor.

## 0. Preparacion

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

from conexion_sql import read_sql
from eda_utils import asegurar_directorio
from modelos_utils import (
    crear_features_temporales,
    crear_lags_por_grupo,
    metricas_regresion,
    preparar_resultados_prediccion,
    train_test_temporal,
)
from visualizacion_utils import guardar_figura, grafico_real_vs_predicho

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

OUT_DATOS = asegurar_directorio(PROJECT_ROOT / "outputs" / "datos")
OUT_GRAFICOS = asegurar_directorio(PROJECT_ROOT / "outputs" / "graficos")

## 1. Dataset de prediccion

Usamos la vista `vw_prediccion_base_ventas`, que ya agrega ventas diarias por canal y categoria e incluye medias moviles.

Objetivo a predecir: `unidades_vendidas`.

In [ ]:
df = read_sql("SELECT * FROM dbo.vw_prediccion_base_ventas")
df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values(["canal", "categoria", "fecha"]).reset_index(drop=True)

print(df.shape)
display(df.head())

## 2. Feature engineering

Creamos variables temporales y retardos.

Ejemplo de logica: las ventas de ayer, de la semana pasada y la media movil de 7/30 dias suelen explicar parte de la demanda futura.

In [ ]:
df_features = crear_features_temporales(df, "fecha")
df_features = crear_lags_por_grupo(
    df_features,
    grupo_cols=["canal", "categoria"],
    objetivo="unidades_vendidas",
    lags=[1, 7, 14, 30],
    ventanas=[7, 14, 30],
)

# Eliminamos filas sin historico suficiente para lags.
df_modelo = df_features.dropna().copy()

print(df_modelo.shape)
display(df_modelo.head())
df_modelo.to_csv(OUT_DATOS / "dataset_ml_ventas.csv", index=False)

## 3. Train/test temporal

No mezclamos aleatoriamente porque seria hacer trampa temporal. Entrenamos con el pasado y validamos con fechas posteriores.

In [ ]:
train, test = train_test_temporal(df_modelo, "fecha", test_ratio=0.2)

objetivo = "unidades_vendidas"
features_numericas = [
    "anio",
    "mes_numero",
    "dia_semana",
    "es_fin_de_semana",
    "dias_desde_inicio",
    "media_movil_7d_unidades",
    "media_movil_30d_unidades",
    "unidades_vendidas_lag_1",
    "unidades_vendidas_lag_7",
    "unidades_vendidas_lag_14",
    "unidades_vendidas_lag_30",
    "unidades_vendidas_media_7",
    "unidades_vendidas_media_14",
    "unidades_vendidas_media_30",
]
features_categoricas = ["canal", "categoria"]
features = features_numericas + features_categoricas

X_train = train[features]
y_train = train[objetivo]
X_test = test[features]
y_test = test[objetivo]

print("train", train["fecha"].min(), train["fecha"].max(), X_train.shape)
print("test", test["fecha"].min(), test["fecha"].max(), X_test.shape)

## 4. Baseline simple

Antes de usar ML, probamos una prediccion sencilla: la media movil de 7 dias. Si un modelo complejo no mejora esto, no aporta valor.

In [ ]:
baseline_pred = test["media_movil_7d_unidades"].astype(float).values
metricas = []
metricas.append({"modelo": "Baseline media movil 7d", **metricas_regresion(y_test, baseline_pred)})

resultados_pred = [
    preparar_resultados_prediccion(
        test[["fecha", "canal", "categoria"]],
        y_test,
        baseline_pred,
        "Baseline media movil 7d",
    )
]

pd.DataFrame(metricas)

## 5. Modelos de machine learning

Probamos modelos interpretables/robustos:

- Regresion lineal: referencia simple.
- Ridge: lineal con regularizacion.
- Random Forest: captura relaciones no lineales.
- Gradient Boosting: suele funcionar bien en datos tabulares.

In [ ]:
preprocesador = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), features_numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore"), features_categoricas),
    ]
)

modelos = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=120, random_state=42, n_jobs=-1, min_samples_leaf=3),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

modelos_entrenados = {}

for nombre, estimador in modelos.items():
    pipeline = Pipeline([("preprocesador", preprocesador), ("modelo", estimador)])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    pred = np.clip(pred, 0, None)

    modelos_entrenados[nombre] = pipeline
    metricas.append({"modelo": nombre, **metricas_regresion(y_test, pred)})
    resultados_pred.append(
        preparar_resultados_prediccion(
            test[["fecha", "canal", "categoria"]],
            y_test,
            pred,
            nombre,
        )
    )

metricas_ml = pd.DataFrame(metricas).sort_values("RMSE")
display(metricas_ml)
metricas_ml.to_csv(OUT_DATOS / "metricas_ml_ventas.csv", index=False)

predicciones_ml = pd.concat(resultados_pred, ignore_index=True)
predicciones_ml.to_csv(OUT_DATOS / "predicciones_ml_ventas.csv", index=False)

## 6. Comparacion visual del mejor modelo

Agregamos por fecha para ver si el modelo acompana la tendencia general.

In [ ]:
mejor_modelo = metricas_ml.iloc[0]["modelo"]
mejores_predicciones = predicciones_ml[predicciones_ml["modelo"] == mejor_modelo].copy()
serie_pred = mejores_predicciones.groupby("fecha", as_index=False).agg(real=("real", "sum"), prediccion=("prediccion", "sum"))

fig, ax = grafico_real_vs_predicho(
    serie_pred,
    titulo=f"Unidades vendidas reales vs predichas - {mejor_modelo}",
)
guardar_figura("ml_real_vs_predicho_mejor_modelo.png", OUT_GRAFICOS)
plt.show()

print("Mejor modelo:", mejor_modelo)

## 7. Importancia de variables

Si el mejor modelo es de arboles, podemos ver que variables pesan mas. Esto ayuda a explicar el modelo sin convertirlo en caja negra.

In [ ]:
modelo_arbol = modelos_entrenados.get(mejor_modelo)

try:
    estimador = modelo_arbol.named_steps["modelo"]
    if hasattr(estimador, "feature_importances_"):
        nombres_features = modelo_arbol.named_steps["preprocesador"].get_feature_names_out()
        importancias = pd.DataFrame({"variable": nombres_features, "importancia": estimador.feature_importances_})
        importancias = importancias.sort_values("importancia", ascending=False).head(25)
        display(importancias)
        importancias.to_csv(OUT_DATOS / "ml_importancia_variables.csv", index=False)

        plt.figure(figsize=(10, 8))
        sns.barplot(data=importancias, x="importancia", y="variable", color="#4C78A8")
        plt.title(f"Importancia de variables - {mejor_modelo}")
        guardar_figura("ml_importancia_variables.png", OUT_GRAFICOS)
        plt.show()
    else:
        print("El mejor modelo no tiene feature_importances_.")
except Exception as exc:
    print("No se pudo calcular importancia de variables:", exc)

## 8. Predicciones para Power BI

Dejamos un CSV con predicciones por fecha, canal y categoria. Mas adelante se podra cargar en Power BI o escribir de vuelta a SQL Server.

In [ ]:
display(predicciones_ml.head())
print(OUT_DATOS / "predicciones_ml_ventas.csv")

## 9. Conclusion para la memoria

Despues de ejecutar, documentar:

- Cual es el baseline y por que era necesario.
- Que modelo mejora mas las metricas.
- Si el error es aceptable para planificacion de stock/promociones.
- Que variables explican mas la prediccion.
- Como se usarian estas predicciones en el dashboard final.